# Setup: Environment & Dependencies

This notebook sets up Ollama and all Python dependencies on Google Colab for **bank transaction intelligence fine-tuning**.

The model will learn to categorize transactions, detect fraud patterns, provide spending insights, and answer customer questions about banking operations.

**Run this first before any other notebooks.**

Before running, make sure the Colab runtime has a GPU attached: `Runtime > Change runtime type > T4 GPU`.

Clone the repository so the Colab runtime has access to all its files (`pyproject.toml`, `src/`, `config/`, etc.), then move into it. All later cells in this notebook assume the working directory is the repository root.

In [1]:
import os
import subprocess

REPO_URL = "https://github.com/Elkinmt19/workshop-databricks-ollama-finetuning.git"
REPO_DIR = "/content/workshop-databricks-ollama-finetuning"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Cloning {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already present at {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

Repository already present at /content/workshop-databricks-ollama-finetuning
Working directory: /content/workshop-databricks-ollama-finetuning


Check that your Colab runtime has GPU compute available.

In [3]:
import subprocess

try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=10
    )
    if result.returncode == 0:
        print("GPU information:")
        print(result.stdout)
        gpu_info = result.stdout.strip().split(',')
        print(f"GPU type: {gpu_info[0].strip()}")
    else:
        print("No GPU detected. Go to Runtime > Change runtime type and select a GPU.")
        print(f"Error: {result.stderr}")
except FileNotFoundError:
    print("nvidia-smi not found. Go to Runtime > Change runtime type and select a GPU.")

GPU information:
Tesla T4, 15360 MiB, 13652 MiB

GPU type: Tesla T4


Install Ollama if not already present.

In [4]:
import subprocess

result = subprocess.run(['which', 'ollama'], capture_output=True, text=True)

if result.returncode == 0:
    print(f"Ollama already installed at: {result.stdout.strip()}")
    version = subprocess.run(['ollama', '--version'], capture_output=True, text=True)
    print(f"Version: {version.stdout.strip()}")
else:
    # install.sh unpacks a .tar.zst archive, which requires zstd. Colab images
    # don't ship it by default, so the install silently dies during extraction
    # without zstd present.
    print("Installing zstd (required by install.sh to unpack the release archive)...")
    subprocess.run(
        ["apt-get", "-y", "-qq", "install", "zstd"],
        capture_output=True, text=True
    )

    print("Installing Ollama...")
    install_cmd = "curl -fsSL https://ollama.ai/install.sh | sh"
    install_result = subprocess.run(
        install_cmd, shell=True, capture_output=True, text=True, timeout=300
    )

    # install.sh also tries to register a systemd service, which fails on Colab
    # (no systemd) even though the binary itself installs correctly. Check for
    # the binary rather than trusting the installer's exit code.
    check = subprocess.run(['which', 'ollama'], capture_output=True, text=True)

    if check.returncode == 0:
        print("Ollama installed successfully.")
        version = subprocess.run(['ollama', '--version'], capture_output=True, text=True)
        print(f"Version: {version.stdout.strip()}")
        if install_result.returncode != 0:
            print("Note: install.sh reported a non-fatal error (likely the systemd step, expected on Colab).")
    else:
        print("Failed to install Ollama.")
        print("--- install.sh stdout ---")
        print(install_result.stdout)
        print("--- install.sh stderr ---")
        print(install_result.stderr)

Ollama already installed at: /usr/local/bin/ollama
Version: Warning: could not connect to a running Ollama instance


Install Python dependencies with `uv`, using the repository's `pyproject.toml` (cloned in the first cell above).

In [5]:
import subprocess
import sys
import os

if not os.path.isfile("pyproject.toml"):
    raise RuntimeError(
        f"pyproject.toml not found in {os.getcwd()}. Re-run the clone cell above first."
    )

print("Installing uv...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

print("Installing Python dependencies from pyproject.toml with uv...")
result = subprocess.run(
    ["uv", "pip", "install", "--system", "-r", "pyproject.toml"],
    capture_output=True, text=True
)

if result.returncode == 0:
    print("All Python dependencies installed.")
else:
    print("Failed to install dependencies.")
    print(result.stderr)
    raise RuntimeError("uv pip install failed. See output above.")

Installing uv...
Installing Python dependencies from pyproject.toml with uv...
All Python dependencies installed.


`uv` just upgraded `numpy` (and other packages) on disk, but Colab's kernel process already has the old `numpy` C extension loaded in memory. Python cannot reload a compiled extension module in the same process, so every subsequent `import numpy` (directly, or transitively via `torch`/`pandas`/`transformers`/`peft`) would raise `ImportError: cannot load module more than once per process`. Restarting the kernel now (equivalent to Databricks' `dbutils.library.restartPython()`) starts a fresh process that imports the newly installed versions cleanly.

**After this cell runs, the kernel restarts and this notebook stops executing.** Manually re-run all cells from the top (`Runtime > Run all`, or run cell-by-cell starting at the clone cell) to continue -- the repo clone and installed packages persist across the restart since they're already on disk.

In [ ]:
import os

os.kill(os.getpid(), 9)

Verify all dependencies are properly installed.

In [ ]:
import sys

print("=== Environment Verification ===")
print(f"Python: {sys.version}")
print()

packages_to_check = [
    ("torch", "PyTorch"),
    ("transformers", "Transformers"),
    ("peft", "PEFT"),
    ("pydantic", "Pydantic"),
    ("pandas", "Pandas"),
]

for module_name, display_name in packages_to_check:
    try:
        mod = __import__(module_name)
        version = getattr(mod, '__version__', 'unknown')
        print(f"{display_name:15} {version}")
    except ImportError:
        print(f"{display_name:15} NOT FOUND")

print()
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Start the Ollama service in the background.

In [7]:
import os
import subprocess
import time
import requests

try:
    response = requests.get("http://localhost:11434/api/tags", timeout=2)
    if response.status_code == 200:
        print("Ollama is already running on port 11434")
except requests.exceptions.RequestException:
    print("Starting Ollama service...")
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        preexec_fn=os.setpgrp
    )

    print("Waiting for Ollama to start...")
    for i in range(30):
        time.sleep(1)
        try:
            response = requests.get("http://localhost:11434/api/tags", timeout=2)
            if response.status_code == 200:
                print(f"Ollama started successfully (took {i + 1} seconds)")
                break
        except requests.exceptions.RequestException:
            if i % 5 == 0:
                print(f"  Waiting... {i}s")
    else:
        print("Ollama may still be starting. Check status in the next cell.")

print("\nOllama available at http://localhost:11434")

models_result = subprocess.run(['ollama', 'list'], capture_output=True, text=True)
print("\nRegistered models:")
print(models_result.stdout or "  (none yet -- fine-tuned model will appear after notebook 03)")

Starting Ollama service...
Waiting for Ollama to start...
  Waiting... 0s
Ollama started successfully (took 2 seconds)

Ollama available at http://localhost:11434

Registered models:
NAME                          ID              SIZE      MODIFIED       
tinyllama-finetuned:latest    1e9560db5386    1.2 GB    18 minutes ago    



This workshop fine-tunes TinyLlama-1.1B-Chat on **synthetic bank transaction data** (categorization, fraud detection, spending analysis, customer support Q&A) using Hugging Face `transformers`/`peft` with LoRA. Ollama is used later (notebook 03) to serve the fine-tuned model after GGUF export. The next cell downloads TinyLlama's HF weights (~2.2GB) so they're cached for notebook 03.

Colab runtimes are ephemeral: the cache is lost if the runtime disconnects or recycles. If you want the download to persist across sessions, mount Google Drive first (`from google.colab import drive; drive.mount('/content/drive')`) and set `HF_HOME` to a path under `/content/drive`. This notebook assumes a single continuous session and does not mount Drive by default.

In [ ]:
import sys

# Block flash_attn to avoid an ABI mismatch with the pip-installed torch build
for _mod in list(sys.modules.keys()):
    if "flash_attn" in _mod:
        del sys.modules[_mod]
sys.modules["flash_attn"] = None
sys.modules["flash_attn_2_cuda"] = None

from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Downloading/caching {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, attn_implementation="eager")
print(f"\n{MODEL_ID} cached locally ({sum(p.numel() for p in model.parameters()) / 1e9:.2f}B params)")

Your environment is now ready for **bank transaction intelligence fine-tuning**.

## Next steps

1. **Data exploration**: run `02_data_exploration` -- generates and explores synthetic bank transaction training data
2. **Fine-tuning**: run `03_finetuning` -- LoRA fine-tunes TinyLlama on transaction intelligence tasks
3. **Evaluation**: run `04_evaluation` -- compares the fine-tuned model against the baseline on transaction understanding

## Training domain

The model learns to handle:
- **Transaction categorization** -- classify purchases (groceries, transport, subscriptions, etc.)
- **Fraud detection** -- analyze transactions for suspicious patterns
- **Spending insights** -- summarize and advise on monthly spending
- **Customer support** -- answer banking questions about charges, refunds, transfers

## Environment details

- Python version: 3.12+
- GPU: Colab T4 (or better)
- PyTorch: installed with CUDA support
- Ollama: running on localhost:11434 (used for serving in notebook 03)
- TinyLlama-1.1B-Chat: base model cached locally for this session

## Troubleshooting

- If `nvidia-smi` reports no GPU, go to `Runtime > Change runtime type` and select a GPU, then re-run this notebook.
- If the model cache is gone in a later notebook, the Colab runtime likely disconnected or recycled -- re-run this notebook, or mount Google Drive to persist the cache across sessions.
- If Ollama fails to start, re-run the "Start the Ollama service" cell; check `ollama --version` and that port 11434 is free.